
## Text Pre-processing and Representations

## Imports

In [2]:
import numpy as np
import pandas as pd
import math
import re
from collections import Counter

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 1:Tokenization and Pre-processing



In [3]:
corpus = [
    "The Cat chased the Mouse.",
    "The dog Barked at the Cat.",
    "The mouse ran away from the Cat."
]

stop_words = {"the", "at", "from", "away"}

def preprocess(text, remove_stopwords=False):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    return tokens

# Tokenize
tokenized_docs = [preprocess(doc, remove_stopwords=False) for doc in corpus]

for i, tokens in enumerate(tokenized_docs, 1):
    print(f"D{i} tokens: {tokens}")

D1 tokens: ['the', 'cat', 'chased', 'the', 'mouse']
D2 tokens: ['the', 'dog', 'barked', 'at', 'the', 'cat']
D3 tokens: ['the', 'mouse', 'ran', 'away', 'from', 'the', 'cat']


## Vocabulary Construction



In [5]:
all_words = [word for doc in tokenized_docs for word in doc]
vocabulary = sorted(set(all_words))

print("Vocabulary (Alphabetically sorted!):")
print(vocabulary)
print(f"\nVocabulary size: {len(vocabulary)}")

Vocabulary (Alphabetically sorted!):
['at', 'away', 'barked', 'cat', 'chased', 'dog', 'from', 'mouse', 'ran', 'the']

Vocabulary size: 10


## Term Frequency (TF) Calculation


$$TF_{t,d} = \frac{f_{t,d}}{\sum_k f_{k,d}}$$

where $f_{t,d}$ is the frequency of term $t$ in document $d$, and the denominator is the total number of terms in $d$.

In [6]:
def compute_tf(tokens, vocab):
    counts = Counter(tokens)
    total = len(tokens)
    return {word: counts.get(word, 0) / total for word in vocab}

tf_list = [compute_tf(doc, vocabulary) for doc in tokenized_docs]

tf_df = pd.DataFrame(tf_list, index=["D1", "D2", "D3"], columns=vocabulary)
print("Term Frequency Matrix:")
print(tf_df.to_string())

Term Frequency Matrix:
          at      away    barked       cat  chased       dog      from     mouse       ran       the
D1  0.000000  0.000000  0.000000  0.200000     0.2  0.000000  0.000000  0.200000  0.000000  0.400000
D2  0.166667  0.000000  0.166667  0.166667     0.0  0.166667  0.000000  0.000000  0.000000  0.333333
D3  0.000000  0.142857  0.000000  0.142857     0.0  0.000000  0.142857  0.142857  0.142857  0.285714


## Normalized TF

Normalize the TF using:

$$w_{tf}(t,d) = \begin{cases} 1 + \log(tf_{t,d}) & \text{if } tf_{t,d} > 0 \\ 0 & \text{otherwise} \end{cases}$$

In [7]:
def normalized_tf(tokens, vocab):
    counts = Counter(tokens)
    total = len(tokens)
    result = {}
    for word in vocab:
        raw_tf = counts.get(word, 0) / total
        result[word] = (1 + math.log10(raw_tf)) if raw_tf > 0 else 0
    return result

norm_tf_list = [normalized_tf(doc, vocabulary) for doc in tokenized_docs]

norm_tf_df = pd.DataFrame(norm_tf_list, index=["D1", "D2", "D3"], columns=vocabulary).round(4)
print("Normalized TF Matrix:")
print(norm_tf_df.to_string())

Normalized TF Matrix:
        at    away  barked     cat  chased     dog    from   mouse     ran     the
D1  0.0000  0.0000  0.0000  0.3010   0.301  0.0000  0.0000  0.3010  0.0000  0.6021
D2  0.2218  0.0000  0.2218  0.2218   0.000  0.2218  0.0000  0.0000  0.0000  0.5229
D3  0.0000  0.1549  0.0000  0.1549   0.000  0.0000  0.1549  0.1549  0.1549  0.4559


## Inverse Document Frequency (IDF) Calculation

$$IDF(t) = \log_{10}\left(\frac{N}{1 + df_t}\right)$$

where $N$ is the total number of documents, and $df_t$ is the number of documents containing term $t$.

In [ ]:
def compute_idf(tokenized_docs, vocab):
    """Compute IDF for each term in vocabulary."""
    N = len(tokenized_docs)
    idf = {}
    for word in vocab:
        df = sum(1 for doc in tokenized_docs if word in doc)
        idf[word] = math.log10(N / (1 + df))
    return idf

idf = compute_idf(tokenized_docs, vocabulary)

idf_df = pd.DataFrame([idf], index=["IDF"], columns=vocabulary).round(4)
print("IDF Values for each term:")
print(idf_df.to_string())

IDF Values for each term:
         at    away  barked     cat  chased     dog    from  mouse     ran     the
IDF  0.1761  0.1761  0.1761 -0.1249  0.1761  0.1761  0.1761    0.0  0.1761 -0.1249


## Compute TF-IDF Vectors

$$TF\text{-}IDF(t, d) = TF_{t,d} \times IDF(t)$$

In [ ]:
def compute_tfidf(norm_tf_list, idf, vocab):
    """Compute TF-IDF matrix."""
    tfidf_list = []
    for tf in norm_tf_list:
        tfidf_list.append({word: round(tf[word] * idf[word], 4) for word in vocab})
    return tfidf_list

tfidf_list = compute_tfidf(norm_tf_list, idf, vocabulary)

tfidf_df = pd.DataFrame(tfidf_list, index=["D1", "D2", "D3"], columns=vocabulary)
print("TF-IDF Matrix:")
print(tfidf_df.to_string())

TF-IDF Matrix:
        at    away  barked     cat  chased     dog    from  mouse     ran     the
D1  0.0000  0.0000  0.0000 -0.0376   0.053  0.0000  0.0000    0.0  0.0000 -0.0752
D2  0.0391  0.0000  0.0391 -0.0277   0.000  0.0391  0.0000    0.0  0.0000 -0.0653
D3  0.0000  0.0273  0.0000 -0.0194   0.000  0.0000  0.0273    0.0  0.0273 -0.0570


## Exercise 1

**Q1. What is the intuition behind using TF-IDF instead of raw term frequency?**

Term frequency only tells how many times a word appears, but it doesn’t show if the word is important. Common words like “the” appear everywhere but don’t mean much. TF-IDF reduces the weight of such common words and gives more importance to words that are unique to a document.

**Q2. Which terms receive higher importance in the TF-IDF representations?**

Words that appear many times in one document but are rare in other documents get higher importance. For example, words like “chased,” “barked,” and “ran” are more important because they are not common in all documents.

**Q3. How would the representation change if you added more documents to the corpus?**

When more documents are added, some words may appear more often across documents, so their importance decreases. Rare words still remain important. Also, new words can be added, which increases the overall vocabulary.

## Verification using Scikit-learn TfidfVectorizer

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

sklearn_df = pd.DataFrame(
    X.toarray(),
    index=["D1", "D2", "D3"],
    columns=vectorizer.get_feature_names_out()
).round(4)

print("Scikit-learn TF-IDF Matrix (for reference):")
print(sklearn_df.to_string())
print("\nNote: sklearn uses slightly different normalization (L2 norm + smooth_idf=True by default).")

Scikit-learn TF-IDF Matrix (for reference):
        at    away  barked     cat  chased     dog    from   mouse     ran     the
D1  0.0000  0.0000  0.0000  0.3240  0.5486  0.0000  0.0000  0.4172  0.0000  0.6480
D2  0.4591  0.0000  0.4591  0.2712  0.0000  0.4591  0.0000  0.0000  0.0000  0.5423
D3  0.0000  0.4335  0.0000  0.2560  0.0000  0.0000  0.4335  0.3297  0.4335  0.5120

Note: sklearn uses slightly different normalization (L2 norm + smooth_idf=True by default).



# Exercise 2: Data Generation for CBOW (word2vec)

**Objective:** Construct look-up tables with context–target pairs for the Continuous Bag of Words (CBOW) model.


## Vocabulary and Indexing



In [9]:
# Corpus after stop-word removal
docs_cbow = [
    ["cat", "chased", "mouse"],
    ["dog", "barked", "cat"],
    ["mouse", "ran", "cat"]
]

# Build vocabulary
vocab_cbow = sorted(set(w for doc in docs_cbow for w in doc))
word2idx = {w: i for i, w in enumerate(vocab_cbow)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab_cbow)  # vocabulary size

print(f"Vocabulary ({V} words): {vocab_cbow}")
print(f"\nWord → Index mapping: {word2idx}")

def one_hot(word, word2idx, V):
    vec = [0] * V
    vec[word2idx[word]] = 1
    return vec

print("\nWord Index Table with One-Hot Vectors:")
oh_data = [(w, word2idx[w], one_hot(w, word2idx, V)) for w in vocab_cbow]
oh_df = pd.DataFrame(oh_data, columns=["Word", "Index", "One-Hot Vector"])
print(oh_df.to_string(index=False))

Vocabulary (6 words): ['barked', 'cat', 'chased', 'dog', 'mouse', 'ran']

Word → Index mapping: {'barked': 0, 'cat': 1, 'chased': 2, 'dog': 3, 'mouse': 4, 'ran': 5}

Word Index Table with One-Hot Vectors:
  Word  Index     One-Hot Vector
barked      0 [1, 0, 0, 0, 0, 0]
   cat      1 [0, 1, 0, 0, 0, 0]
chased      2 [0, 0, 1, 0, 0, 0]
   dog      3 [0, 0, 0, 1, 0, 0]
 mouse      4 [0, 0, 0, 0, 1, 0]
   ran      5 [0, 0, 0, 0, 0, 1]


## Create Look-Up Tables with Context



In [11]:
def build_lookup_table(doc, word2idx, window=1):
    pairs = []
    for i, target in enumerate(doc):
        context = []
        for j in range(i - window, i + window + 1):
            if j != i and 0 <= j < len(doc):
                context.append(doc[j])
        context_idx = [word2idx[w] for w in context]
        pairs.append({
            "Context (Words)": context,
            "Context (Index)": context_idx,
            "Target Word": target,
            "Target (Index)": word2idx[target]
        })
    return pairs

for doc_idx, doc in enumerate(docs_cbow, 1):
    pairs = build_lookup_table(doc, word2idx, window=1)
    df = pd.DataFrame(pairs)
    print(f"\n Look-Up Table for D{doc_idx}: {doc}")
    print(df.to_string(index=False))


 Look-Up Table for D1: ['cat', 'chased', 'mouse']
Context (Words) Context (Index) Target Word  Target (Index)
       [chased]             [2]         cat               1
   [cat, mouse]          [1, 4]      chased               2
       [chased]             [2]       mouse               4

 Look-Up Table for D2: ['dog', 'barked', 'cat']
Context (Words) Context (Index) Target Word  Target (Index)
       [barked]             [0]         dog               3
     [dog, cat]          [3, 1]      barked               0
       [barked]             [0]         cat               1

 Look-Up Table for D3: ['mouse', 'ran', 'cat']
Context (Words) Context (Index) Target Word  Target (Index)
          [ran]             [5]       mouse               4
   [mouse, cat]          [4, 1]         ran               5
          [ran]             [5]         cat               1


## Exercise 2 (CBOW)

**Q1. How does CBOW handle words like "cat" that appear in multiple documents with different contexts?**

> In CBOW, the model takes the context words and averages their embeddings to predict the target word. For example, “cat” appears with different context words in different documents, and each time it helps update the model. During training, these repeated updates slowly adjust the embedding of “cat” so it works well for all its contexts. In the end, the embedding becomes a balanced representation that reflects the overall meaning of “cat” based on all the words it appears with.

**Q2. How does this connect to the Distributional Hypothesis?**

> The Distributional Hypothesis means that a word’s meaning depends on the words around it. CBOW uses this idea by predicting a word based on its surrounding words. So, words that appear in similar contexts, like “cat” and “dog,” get similar updates during training. Because of this, their embeddings become close to each other, showing they have similar meanings.


# Exercise 2 : CBOW Architecture Design and Training


## Define CBOW Architecture

- **Input Layer:** V = 6 neurons (one per vocabulary word, one-hot encoded)
- **Hidden Layer:** N = 4 neurons (embedding dimension, no non-linear activation)
- **Output Layer:** V = 6 neurons (one per vocabulary word, softmax activation)

In [12]:
V = 6
N = 4

print(f"Input Layer  : {V} neurons (one-hot, size = |V| = {V})")
print(f"Hidden Layer : {N} neurons (embedding dimension N = {N})")
print(f"Output Layer : {V} neurons (= |V|, one per vocabulary word)")

Input Layer  : 6 neurons (one-hot, size = |V| = 6)
Hidden Layer : 4 neurons (embedding dimension N = 4)
Output Layer : 6 neurons (= |V|, one per vocabulary word)


## Weight Matrix Dimensions and Random Initialization

- **W_input** (Embedding Matrix): shape $|V| \times N$ = $6 \times 4$  
- **W_hidden** (Output Weight Matrix): shape $N \times |V|$ = $4 \times 6$

In [13]:
W_input = np.array([
    [ 0.1,  0.2, -0.1,  0.0],
    [ 0.4,  0.5, -0.1,  0.2],
    [-0.2,  0.3,  0.2, -0.1],
    [ 0.3, -0.1,  0.1,  0.0],
    [ 0.5, -0.4,  0.1,  0.1],
    [-0.3,  0.0, -0.2,  0.3],
])

W_hidden = np.array([
    [ 0.2, -0.1,  0.3,  0.1,  0.4,  0.0],
    [ 0.1,  0.3, -0.2, -0.1,  0.2,  0.5],
    [-0.4,  0.5,  0.1, -0.2, -0.1,  0.3],
    [ 0.1, -0.2, -0.4,  0.3, -0.3,  0.1],
])

print(f"W_input shape  : {W_input.shape}  (|V| x N = 6 x 4)")
print(f"W_hidden shape : {W_hidden.shape}  (N x |V| = 4 x 6)")
print("\nW_input (Embedding Matrix):")
print(pd.DataFrame(W_input, index=vocab_cbow, columns=[f"N{i}" for i in range(N)]))
print("\nW_hidden (Output Weight Matrix):")
print(pd.DataFrame(W_hidden, index=[f"N{i}" for i in range(N)], columns=vocab_cbow))

W_input shape  : (6, 4)  (|V| x N = 6 x 4)
W_hidden shape : (4, 6)  (N x |V| = 4 x 6)

W_input (Embedding Matrix):
         N0   N1   N2   N3
barked  0.1  0.2 -0.1  0.0
cat     0.4  0.5 -0.1  0.2
chased -0.2  0.3  0.2 -0.1
dog     0.3 -0.1  0.1  0.0
mouse   0.5 -0.4  0.1  0.1
ran    -0.3  0.0 -0.2  0.3

W_hidden (Output Weight Matrix):
    barked  cat  chased  dog  mouse  ran
N0     0.2 -0.1     0.3  0.1    0.4  0.0
N1     0.1  0.3    -0.2 -0.1    0.2  0.5
N2    -0.4  0.5     0.1 -0.2   -0.1  0.3
N3     0.1 -0.2    -0.4  0.3   -0.3  0.1


## Computations at Hidden Layer


In [14]:
context_words = ["cat", "mouse"]
target_word   = "chased"

print(f"Context: {context_words}  |  Target: {target_word}")

projections = {}
for word in context_words:
    idx = word2idx[word]
    one_hot_vec = np.zeros(V)
    one_hot_vec[idx] = 1
    proj = W_input[idx]
    projections[word] = proj
    print(f"\nProjection for '{word}' (index {idx}):")
    print(f"  One-hot: {one_hot_vec.astype(int).tolist()}")
    print(f"  → Selects row {idx} of W_input: {proj}")

# Average context embeddings
V_hat = np.mean(list(projections.values()), axis=0)
print(f"\nAverage context vector (hidden layer V̂):")
print(f"  ({projections['cat']} + {projections['mouse']}) / 2")
print(f"  = {V_hat}")

Context: ['cat', 'mouse']  |  Target: chased

Projection for 'cat' (index 1):
  One-hot: [0, 1, 0, 0, 0, 0]
  → Selects row 1 of W_input: [ 0.4  0.5 -0.1  0.2]

Projection for 'mouse' (index 4):
  One-hot: [0, 0, 0, 0, 1, 0]
  → Selects row 4 of W_input: [ 0.5 -0.4  0.1  0.1]

Average context vector (hidden layer V̂):
  ([ 0.4  0.5 -0.1  0.2] + [ 0.5 -0.4  0.1  0.1]) / 2
  = [0.45 0.05 0.   0.15]


### Project to Output Layer

Multiply the average context vector $\hat{V}$ by $W_{hidden}$ to get raw output logits.

In [15]:
output_logits = V_hat @ W_hidden

print("Output logits (V̂ × W_hidden):")
for word, score in zip(vocab_cbow, output_logits):
    print(f"  {word:8s} → {score:.4f}")

Output logits (V̂ × W_hidden):
  barked   → 0.1100
  cat      → -0.0600
  chased   → 0.0650
  dog      → 0.0850
  mouse    → 0.1450
  ran      → 0.0400


## The Output Layer: Softmax

The output logits are passed through **softmax** to produce a probability distribution over the vocabulary:

$$P(\text{word}_i \mid \text{context}) = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$


In [17]:
def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

probs = softmax(output_logits)

print("Softmax Probabilities:")
result_df = pd.DataFrame({
    "Word": vocab_cbow,
    "Logit (z)": output_logits.round(4),
    "P(word | context)": probs.round(4)
})
print(result_df.to_string(index=False))

predicted = vocab_cbow[np.argmax(probs)]
actual    = target_word
print(f"\nPredicted word : '{predicted}' (highest probability = {probs.max():.4f})")
print(f"Actual target  : '{actual}'")
print(f"Correct        : {predicted == actual}  (weights are random — training needed!)")

Softmax Probabilities:
  Word  Logit (z)  P(word | context)
barked      0.110             0.1741
   cat     -0.060             0.1469
chased      0.065             0.1665
   dog      0.085             0.1698
 mouse      0.145             0.1803
   ran      0.040             0.1624

Predicted word : 'mouse' (highest probability = 0.1803)
Actual target  : 'chased'
Correct        : False  (weights are random — training needed!)


## Full Forward Pass: All Context-Target Pairs


In [19]:
def cbow_forward(context_words, W_input, W_hidden, word2idx, idx2word, vocab):
    context_embeds = np.array([W_input[word2idx[w]] for w in context_words])

    v_hat = context_embeds.mean(axis=0)

    logits = v_hat @ W_hidden

    probs = softmax(logits)
    return v_hat, logits, probs

print("CBOW Forward Pass: All Context-Target Pairs")

for doc_idx, doc in enumerate(docs_cbow, 1):
    pairs = build_lookup_table(doc, word2idx, window=1)
    print(f"\nnDocument {doc_idx}: {doc}")
    for pair in pairs:
        ctx  = pair["Context (Words)"]
        tgt  = pair["Target Word"]
        v_hat, logits, probs = cbow_forward(ctx, W_input, W_hidden, word2idx, idx2word, vocab_cbow)
        pred = vocab_cbow[np.argmax(probs)]
        print(f"  Context: {ctx!s:30s} Target: '{tgt}'  Predicted: '{pred}'  P(target)={probs[word2idx[tgt]]:.4f}")

CBOW Forward Pass: All Context-Target Pairs

nDocument 1: ['cat', 'chased', 'mouse']
  Context: ['chased']                     Target: 'cat'  Predicted: 'cat'  P(target)=0.2029
  Context: ['cat', 'mouse']               Target: 'chased'  Predicted: 'mouse'  P(target)=0.1665
  Context: ['chased']                     Target: 'mouse'  Predicted: 'cat'  P(target)=0.1596

nDocument 2: ['dog', 'barked', 'cat']
  Context: ['barked']                     Target: 'dog'  Predicted: 'mouse'  P(target)=0.1619
  Context: ['dog', 'cat']                 Target: 'barked'  Predicted: 'mouse'  P(target)=0.1711
  Context: ['barked']                     Target: 'cat'  Predicted: 'mouse'  P(target)=0.1603

nDocument 3: ['mouse', 'ran', 'cat']
  Context: ['ran']                        Target: 'mouse'  Predicted: 'dog'  P(target)=0.1470
  Context: ['mouse', 'cat']               Target: 'ran'  Predicted: 'mouse'  P(target)=0.1624
  Context: ['ran']                        Target: 'cat'  Predicted: 'dog'  P(targe


# Applying NLP Pre-processing to Trump Tweets Dataset


In [ ]:
df_tweets = pd.read_csv("trumptweets_small.csv")
print(f"Dataset shape: {df_tweets.shape}")
print(f"Columns: {list(df_tweets.columns)}")
df_tweets.head(3)

Dataset shape: (41122, 9)
Columns: ['id', 'link', 'content', 'date', 'retweets', 'favorites', 'mentions', 'hashtags', 'geo']


,id,link,content,date,retweets,favorites,mentions,hashtags,geo
0,1698308935,https://twitter.com/realDonaldTrump/status/169...,Be sure to tune in and watch Donald Trump on L...,2009-05-04 20:54:25,500,868,NaN,NaN,NaN
1,1701461182,https://twitter.com/realDonaldTrump/status/170...,Donald Trump will be appearing on The View tom...,2009-05-05 03:00:10,33,273,NaN,NaN,NaN
2,1737479987,https://twitter.com/realDonaldTrump/status/173...,Donald Trump reads Top Ten Financial Tips on L...,2009-05-08 15:38:08,12,18,NaN,NaN,NaN


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tweets = df_tweets['content'].dropna().tolist()
print(f"Total tweets: {len(tweets)}")

tfidf_vec = TfidfVectorizer(max_features=20, stop_words='english')
X_tweets = tfidf_vec.fit_transform(tweets)

tweet_tfidf_df = pd.DataFrame(
    X_tweets.toarray()[:5],
    columns=tfidf_vec.get_feature_names_out()
).round(3)

print("\nTop-20 TF-IDF features for first 5 tweets:")
tweet_tfidf_df

Total tweets: 41122

Top-20 TF-IDF features for first 5 tweets:


,america,bit,com,country,donald,great,http,https,just,ly,new,people,pic,president,realdonaldtrump,thank,thanks,trump,twitter,www
0,0.0,0.0,0.000,0.0,0.798,0.0,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.602,0.0,0.0
1,0.0,0.0,0.000,0.0,0.615,0.0,0.000,0.0,0.0,0.0,0.638,0.0,0.0,0.0,0.0,0.0,0.0,0.464,0.0,0.0
2,0.0,0.0,0.431,0.0,0.610,0.0,0.481,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.460,0.0,0.0
3,0.0,0.0,0.477,0.0,0.000,0.0,0.532,0.0,0.0,0.0,0.699,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0
4,0.0,0.0,0.000,0.0,0.798,0.0,0.000,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.602,0.0,0.0


In [ ]:
mean_tfidf = X_tweets.toarray().mean(axis=0)
top_words_df = pd.DataFrame({
    'Word': tfidf_vec.get_feature_names_out(),
    'Mean TF-IDF': mean_tfidf
}).sort_values('Mean TF-IDF', ascending=False)

print("Top words by Mean TF-IDF score (across all tweets):")
print(top_words_df.to_string(index=False))

Top words by Mean TF-IDF score (across all tweets):
           Word  Mean TF-IDF
realdonaldtrump     0.143906
          great     0.095446
          trump     0.081364
            com     0.074841
           http     0.065453
        twitter     0.056527
      president     0.048682
           just     0.047732
         thanks     0.045824
         people     0.045370
          thank     0.045050
        country     0.034532
            new     0.032132
         donald     0.031718
            pic     0.031303
          https     0.029611
        america     0.029332
             ly     0.026815
            www     0.026093
            bit     0.025605
